In [1]:
import matplotlib.pyplot as plt
import utils_, config, model
import os
import numpy as np
import pandas as pd
import Analysis_function
import preprocess

import warnings
warnings.filterwarnings('ignore')


# 1. Fetch dataset in dict format - Work data / Head data
dataset_Work = preprocess.get_all_data(config.year_list, config.file_names_Work)
dataset_Head = preprocess.get_all_data(config.year_list, config.file_names_Head)

# 2. Check the every column and its semantic name
#utils_.see_col_idx_and_name(dataset['2020']['data'], dataset['2020']['meta'])

# 3. Check the intersection for the number of organization - year by year (e.g. compare 2020 - 2021)
#_ = Analysis_function.compare_company_ids_in_dataset(dataset_Work)

# 4. Check the intersection for the number of organization - All years (2020-2023) - we do this again in next step
#_ = Analysis_function.get_common_company_ids_all_years(dataset_Work)

# 5. Check the intersection for the number of organization (All) and filter; select only the organization that involves throughout all years
dataset_Work = preprocess.filter_dataset_by_common_ids(dataset_Work)

# 6. Target variable check - 1. Type count (value_counts()) / 2. check Nan
Analysis_function.target_variable_check(dataset_Work, target_variable=config.target_col)

# 7. Nan 값 25% 언더면 정수형 평균값으로 넣고, 위면 해당 컬럼 삭제
dataset_Work = preprocess.clean_all_years(dataset_Work, columns_to_drop=[], verbose=True)

# 8. Store only the common columns in each yearly dataset - 컬럼이 다르면 안되니깐.
dataset_Work = preprocess.unify_columns_by_base_name(dataset_Work)

C:\Users\hml76\PycharmProjects\HRD2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HCCP_2ndWave_Work_1st_v2.sav ===> 2020 data
HCCP_2ndWave_Work_2nd_v2.sav ===> 2021 data
HCCP_2ndWave_Work_3rd_v3.sav ===> 2022 data
HCCP_2ndWave_Work_4th.sav ===> 2023 data
HCCP_2ndWave_Head_1st(최종).sav ===> 2020 data
HCCP_2ndWave_Head_2nd(최종).sav ===> 2021 data
HCCP_2ndWave_Head_3rd(최종).sav ===> 2022 data
HCCP_2ndWave_Head_4th.sav ===> 2023 data
공통 기업 ID 개수: 384
2020 필터링 후 행 개수: 7054
2021 필터링 후 행 개수: 7613
2022 필터링 후 행 개수: 7338
2023 필터링 후 행 개수: 8740
2020 - W20Q09A : NaN 0개 / 전체 7054개 (0.00%)
W20Q09A
3.0    2372
8.0    1682
4.0    1534
2.0     873
5.0     434
1.0     159
Name: count, dtype: int64
2021 - W21Q09A : NaN 0개 / 전체 7613개 (0.00%)
W21Q09A
-8.0    2331
 3.0    2318
 4.0    1511
 2.0     907
 5.0     369
 1.0     177
Name: count, dtype: int64
2022 - W22Q09A : NaN 0개 / 전체 7338개 (0.00%)
W22Q09A
3.0    2263
8.0    1922
4.0    1436
2.0     893
5.0     501
1.0     323
Name: count, dtype: int64
2023 - W23Q09A : NaN 0개 / 전체 8740개 (0.00%)
W23Q09A
3.0    4008
4.0    2034
2.0    1133
5.0   

In [2]:
#df_21_21['model20_21_output'] = np.where(y_pred_prob > 0.6, y_pred_prob, 0) # 예측 확률이 0.6 이상인 경우만 feature로 사용, 나머지는 0으로

ACC = {}
for year in ['2020', '2021', '2022', '2023']:
    if year == '2021' or '2023':
        if year == '2021':
            # Merge data - Work (X) + Head (y)
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2021')
            df1 = preprocess.standardize_feature_names(df1)
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            #X['model20_21_output'] = model_20_21.predict_proba(X)[:, 1]
            y_pred_prob = model_20_21.predict_proba(X)[:, 1]
            X['model20_21_output'] = np.where(y_pred_prob > 0.6, y_pred_prob, 0)
            acc, model_21_21 = model.train_evaluate_model(X, y, learning_graph_show=False)


            df2, label_col2 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            df2 = preprocess.standardize_feature_names(df2)
            X, y = preprocess.clean_target_classes(df2, target_col=label_col2)
            #X['model20_23_output'] = model_20_23.predict_proba(X)[:, 1]
            y_pred_prob = model_20_23.predict_proba(X)[:, 1]
            X['model20_23_output'] = np.where(y_pred_prob > 0.6, y_pred_prob, 0)
            acc2, model_21_23 = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC[year] = [acc, acc2]

        elif year == '2023':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            df1 = preprocess.standardize_feature_names(df1)
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            add_20_21 = model_20_21.predict_proba(X)[:, 1]
            add_20_23 = model_20_23.predict_proba(X)[:, 1]

            X_tmp1 = X.copy()
            #X_tmp1['model20_21_output'] = add_20_21
            X_tmp1['model20_21_output'] = np.where(add_20_21 > 0.6, add_20_21, 0)
            add_21_21 = model_21_21.predict_proba(X_tmp1)[:, 1]

            X_tmp2 = X.copy()
            #X_tmp2['model20_23_output'] = add_20_23
            X_tmp2['model20_23_output'] = np.where(add_20_23 > 0.6, add_20_23, 0)
            add_21_23 = model_21_23.predict_proba(X_tmp2)[:, 1]

            #X['model20_21_output'], X['model20_23_output'], X['model21_21_output'], X['model21_23_output'] = add_20_21, add_20_23, add_21_21, add_21_23
            X['model20_21_output'] = np.where(add_20_21 > 0.6, add_20_21, 0)
            X['model20_23_output'] = np.where(add_20_23 > 0.6, add_20_23, 0)
            X['model21_21_output'] = np.where(add_21_21 > 0.6, add_21_21, 0)
            X['model21_23_output'] = np.where(add_21_23 > 0.6, add_21_23, 0)

            #X['model22_23_output'] = model_22_23.predict_proba(X)[:, 1]
            y_pred_prob = model_22_23.predict_proba(X)[:, 1]
            X['model22_23_output'] = np.where(y_pred_prob > 0.6, y_pred_prob, 0)

            acc, _ = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC[year] = acc

    if year == '2020' or '2022':
        if year == '2020':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2021')
            df1 = preprocess.standardize_feature_names(df1)
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            acc, model_20_21 = model.train_evaluate_model(X, y, learning_graph_show=False)


            df2, label_col2 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            df2 = preprocess.standardize_feature_names(df2)
            X, y = preprocess.clean_target_classes(df2, target_col=label_col2)
            acc2, model_20_23 = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC[year] = [acc, acc2]

        elif year == '2022':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            df1 = preprocess.standardize_feature_names(df1)
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            tmp1 = model_20_21.predict_proba(X)[:, 1]
            tmp2 = model_20_23.predict_proba(X)[:, 1]

            X_tmp = X.copy()
            #X_tmp['model20_21_output'] = tmp1
            X_tmp['model20_21_output'] = np.where(tmp1 > 0.6, tmp1, 0)
            tmp3 = model_21_21.predict_proba(X_tmp)[:, 1]

            X_tmp = X.copy()
            #X_tmp['model20_23_output'] = tmp2
            X_tmp['model20_23_output'] = np.where(tmp2 > 0.6, tmp2, 0)
            tmp4 = model_21_23.predict_proba(X_tmp)[:, 1]

            #X['model20_21_output'], X['model20_23_output'], X['model21_21_output'], X['model21_23_output'] = tmp1, tmp2, tmp3, tmp4
            X['model20_21_output'] = np.where(tmp1 > 0.6, tmp1, 0)
            X['model20_23_output'] = np.where(tmp2 > 0.6, tmp2, 0)
            X['model21_21_output'] = np.where(tmp3 > 0.6, tmp3, 0)
            X['model21_23_output'] = np.where(tmp4 > 0.6, tmp4, 0)
            acc, model_22_23 = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC[year] = acc

Year: 2020 | Merged shape: (7054, 109) | work shape: (7054, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1247
0.0     671
Name: count, dtype: int64
Resampled class distribution:
0.0    986
1.0    986
Name: count, dtype: int64
XGBoost Accuracy ========>  78.38541666666666 %
Year: 2020 | Merged shape: (7054, 109) | work shape: (7054, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1388
0.0     309
Name: count, dtype: int64
Resampled class distribution:
1.0    1110
0.0    1110
Name: count, dtype: int64
XGBoost Accuracy ========>  86.1764705882353 %
Year: 2021 | Merged shape: (7613, 109) | work shape: (7613, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1280
0.0     709
Name: count, dtype: int64
Resampled class distribution:
1.0    1002
0.0    1002
Name: count, dtype: int64
XGBoost Accuracy ========>  77.38693467336684 %
Year: 2021 | Merged shape: (7613, 109) | work shape: (7613, 126) | hea

In [3]:
#df_21_21['model20_21_output'] = np.where(y_pred_prob > 0.6, y_pred_prob, 0) # 예측 확률이 0.6 이상인 경우만 feature로 사용, 나머지는 0으로

ACC1 = {}
for year in ['2020', '2021', '2022', '2023']:
    if year == '2021' or '2023':
        if year == '2021':
            # Merge data - Work (X) + Head (y)
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2021')
            df1 = preprocess.standardize_feature_names(df1)
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            X['model20_21_output'] = model_20_21.predict_proba(X)[:, 1]
            acc, model_21_21 = model.train_evaluate_model(X, y, learning_graph_show=False)

            df2, label_col2 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            df2 = preprocess.standardize_feature_names(df2)
            X, y = preprocess.clean_target_classes(df2, target_col=label_col2)
            X['model20_23_output'] = model_20_21.predict_proba(X)[:, 1]

            acc2, model_21_23 = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC1[year] = [acc, acc2]

        elif year == '2023':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            df1 = preprocess.standardize_feature_names(df1)
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            add_20_21 = model_20_21.predict_proba(X)[:, 1]
            add_20_23 = model_20_23.predict_proba(X)[:, 1]

            X_tmp1 = X.copy()
            X_tmp1['model20_21_output'] = add_20_21
            add_21_21 = model_21_21.predict_proba(X_tmp1)[:, 1]

            X_tmp2 = X.copy()
            X_tmp2['model20_23_output'] = add_20_23
            add_21_23 = model_21_23.predict_proba(X_tmp2)[:, 1]

            X['model20_21_output'], X['model20_23_output'], X['model21_21_output'], X['model21_23_output'] = add_20_21, add_20_23, add_21_21, add_21_23
            X['model22_23_output'] = model_22_23.predict_proba(X)[:, 1]

            acc, _ = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC1[year] = acc

    if year == '2020' or '2022':
        if year == '2020':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2021')
            df1 = preprocess.standardize_feature_names(df1)
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            acc, model_20_21 = model.train_evaluate_model(X, y, learning_graph_show=False)


            df2, label_col2 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            df2 = preprocess.standardize_feature_names(df2)
            X, y = preprocess.clean_target_classes(df2, target_col=label_col2)
            acc2, model_20_23 = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC1[year] = [acc, acc2]


        elif year == '2022':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            df1 = preprocess.standardize_feature_names(df1)
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            tmp1 = model_20_21.predict_proba(X)[:, 1]
            tmp2 = model_20_23.predict_proba(X)[:, 1]

            X_tmp = X.copy()
            X_tmp['model20_21_output'] = tmp1
            tmp3 = model_21_21.predict_proba(X_tmp)[:, 1]

            X_tmp = X.copy()
            X_tmp['model20_23_output'] = tmp2
            tmp4 = model_21_23.predict_proba(X_tmp)[:, 1]

            X['model20_21_output'], X['model20_23_output'], X['model21_21_output'], X['model21_23_output'] = tmp1, tmp2, tmp3, tmp4

            acc, model_22_23 = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC1[year] = acc

Year: 2020 | Merged shape: (7054, 109) | work shape: (7054, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1247
0.0     671
Name: count, dtype: int64
Resampled class distribution:
0.0    986
1.0    986
Name: count, dtype: int64
XGBoost Accuracy ========>  78.38541666666666 %
Year: 2020 | Merged shape: (7054, 109) | work shape: (7054, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1388
0.0     309
Name: count, dtype: int64
Resampled class distribution:
1.0    1110
0.0    1110
Name: count, dtype: int64
XGBoost Accuracy ========>  86.1764705882353 %
Year: 2021 | Merged shape: (7613, 109) | work shape: (7613, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1280
0.0     709
Name: count, dtype: int64
Resampled class distribution:
1.0    1000
0.0    1000
Name: count, dtype: int64
XGBoost Accuracy ========>  76.13065326633166 %
Year: 2021 | Merged shape: (7613, 109) | work shape: (7613, 126) | hea

In [4]:
ACC2 = {}

for year in ['2020', '2021', '2022', '2023']:
    if year == '2021' or '2023':
        if year == '2021':
            # Merge data - Work (X) + Head (y)
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2021')
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            acc, _ = model.train_evaluate_model(X, y, learning_graph_show=False)

            df2, label_col2 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            X, y = preprocess.clean_target_classes(df2, target_col=label_col2)
            acc2, _ = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC2[year] = [acc, acc2]

        elif year == '2023':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            acc, _ = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC2[year] = acc

    if year == '2020' or '2022':
        if year == '2020':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2021')
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            acc, _ = model.train_evaluate_model(X, y, learning_graph_show=False)

            df2, label_col2 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            X, y = preprocess.clean_target_classes(df2, target_col=label_col2)
            acc2, _ = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC2[year] = [acc, acc2]

        elif year == '2022':
            df1, label_col1 = preprocess.merge_worker_head_labels(dataset_Work, dataset_Head, year_w=year, year_h='2023')
            X, y = preprocess.clean_target_classes(df1, target_col=label_col1)
            acc, _ = model.train_evaluate_model(X, y, learning_graph_show=False)
            ACC2[year] = acc


Year: 2020 | Merged shape: (7054, 109) | work shape: (7054, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1247
0.0     671
Name: count, dtype: int64
Resampled class distribution:
0.0    986
1.0    986
Name: count, dtype: int64
XGBoost Accuracy ========>  78.38541666666666 %
Year: 2020 | Merged shape: (7054, 109) | work shape: (7054, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1388
0.0     309
Name: count, dtype: int64
Resampled class distribution:
1.0    1110
0.0    1110
Name: count, dtype: int64
XGBoost Accuracy ========>  86.1764705882353 %
Year: 2021 | Merged shape: (7613, 109) | work shape: (7613, 126) | head shape: (500, 2)

🧪 Processing: 
Original class distribution:
1.0    1280
0.0     709
Name: count, dtype: int64
Resampled class distribution:
1.0    999
0.0    999
Name: count, dtype: int64
XGBoost Accuracy ========>  77.63819095477386 %
Year: 2021 | Merged shape: (7613, 109) | work shape: (7613, 126) | head 

In [5]:
ACC # With cumulate / 0.6 prob

{'2020': [0.7838541666666666, 0.861764705882353],
 '2021': [0.7738693467336684, 0.8719512195121951],
 '2022': 0.9065420560747663,
 '2023': 0.9140811455847255}

In [6]:
ACC1 # With cumulate / all prob

{'2020': [0.7838541666666666, 0.861764705882353],
 '2021': [0.7613065326633166, 0.8628048780487805],
 '2022': 0.897196261682243,
 '2023': 0.9140811455847255}

In [7]:
ACC2  #No cumulate

{'2020': [0.7838541666666666, 0.861764705882353],
 '2021': [0.7763819095477387, 0.8567073170731707],
 '2022': 0.881619937694704,
 '2023': 0.9045346062052506}